# AI Resume Screening & Team Ranking System
## Academic Submission - Model Development & Evaluation

### Project Objective
Develop a robust NLP-based system to automate the initial screening of resumes against a given job description (JD). We utilize term-frequency-inverse document frequency (TF-IDF) and Cosine Similarity to calculate matching scores.

### System Components
1. **Preprocessing**: Text cleaning, tokenization, lemmatization via spaCy.
2. **Feature Extraction**: TF-IDF vectorization via Scikit-Learn.
3. **Similarity Calculation**: Cosine Similarity measured in vector space.
4. **Skill Evaluation**: Explicit extraction of matched and missing skills based on a predefined database.

### 1. Setup and Imports

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Add root project to path for utils access
sys.path.append(os.path.abspath(".."))

from utils.extraction import extract_text
from utils.nlp_pipeline import clean_text, extract_skills

print("Libraries loaded successfully.")

### 2. Sample Data Loading (Internal Test)
We'll load some sample resumes from the `data/` folder for this evaluation.

In [ ]:
DATA_PATH = "../data/"
resumes = []
filenames = []

if os.path.exists(DATA_PATH):
    for file in os.listdir(DATA_PATH):
        if file.lower().endswith(('.pdf', '.docx')):
            with open(os.path.join(DATA_PATH, file), "rb") as f:
                content = f.read()
            text = extract_text(file, content)
            if text:
                resumes.append(text)
                filenames.append(file)

print(f"Loaded {len(resumes)} resumes for evaluation.")

### 3. NLP Preprocessing Logic
We define a processing pipeline that cleans data and highlights skills. Lemmatization ensures that 'coding' and 'code' are treated as the same feature.

In [ ]:
sample_text = "I am coding in Python and learning machine learning models like TensorFlow."
cleaned = clean_text(sample_text)
skills = extract_skills(sample_text)
print(f"Original: {sample_text}")
print(f"Cleaned: {cleaned}")
print(f"Skills Extracted: {skills}")

### 4. Ranking Model Implementation
We simulate a typical 'Python Developer' Job Description for this evaluation.

In [ ]:
mock_jd = """
Software Engineer - Python Developer
Requirements: Experience with Python, SQL, and FastAPI/Flask.
Skills: Deep Learning, Machine Learning, and Git/CI-CD knowledge preferred.
Tools: Pandas, Scikit-learn, and React knowledge is a plus.
"""

cleaned_jd = clean_text(mock_jd)
cleaned_resumes = [clean_text(r) for r in resumes]

# Fit Tfidf
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform([cleaned_jd] + cleaned_resumes)

# Compute Cosine Similarity
sim_scores = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:])
scores = [round(float(s) * 100, 2) for s in sim_scores[0]]

results_df = pd.DataFrame({
    "Candidate": filenames,
    "Similarity Score (%)": scores
}).sort_values(by="Similarity Score (%)", ascending=False)

results_df.head(10)

### 5. Evaluation & Performance Analysis
The scores indicate the vector proximity between the JD and each resume. A score of 100% would be a perfect match, while 0% would indicate no shared tokens (after cleaning).

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=results_df.head(10), x="Similarity Score (%)", y="Candidate", palette="viridis")
plt.title("Top 10 Ranked Candidates - Similarity Score Analysis")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

### 6. Insights & Observations
1. **Preprocessing Impact**: Lemmatization significantly improves scores for resumes that use different tenses of keywords (e.g., 'developing' vs. 'develop').
2. **Similarity Scoring**: TF-IDF works best when recruiters use standard terminology. It may penalize candidates who use creative synonym for certain skills.
3. **Skill Gaps**: By computing set difference between JD and resume skills, we provides an explicit 'Skills Missing' metric which adds contextual value to the numerical score.